# Full agent eval: every tool, with memory

The other notebooks isolate one backend. This one is the whole agent — the same
`create_agent` instance the Discord bot runs, imported from `agent.agent`, with
all 10 read-only tools and the shared SQLite checkpointer. Questions asked on the
same `thread_id` remember each other.

Use it to watch tool _selection_ — which backend the model reaches for when
nothing is taken away — and whether it carries context forward.


In [ ]:
from agent.agent import get_agent
from agent.memory import thread_config
from agent.tools import ALL_TOOLS

print(len(ALL_TOOLS), "tools:", ", ".join(t.name for t in ALL_TOOLS))

agent = await get_agent()


### Reading a turn

The agent returns the whole thread; these pull the answer text and the token
cost out of the messages one call added.


In [ ]:
def _text(content) -> str:
    """Coerce an LLM message content (str or content-block list) to plain text."""
    if isinstance(content, str):
        return content.strip()
    return "".join(b.get("text", "") for b in content if isinstance(b, dict)).strip()


def _usage(messages: list) -> dict:
    """Total token counts reported across a batch of LLM turns."""
    total = {"input": 0, "output": 0, "total": 0}
    for m in messages:
        reported = getattr(m, "usage_metadata", None) or {}
        total["input"] += reported.get("input_tokens", 0)
        total["output"] += reported.get("output_tokens", 0)
        total["total"] += reported.get("total_tokens", 0)
    return total


### Running one question

`thread_config` supplies the thread id, exactly as in the docs
(`{"configurable": {"thread_id": ...}}`). The tool budget, tool-error handling,
and context summarization are middleware on the agent itself, so nothing is
configured per call. Tokens are tallied over only the messages this call added,
so a follow-up on the same thread reports its own cost.


In [ ]:
_SEEN: dict[str, int] = {}


async def ask(question: str, thread_id: str = "notebook") -> str:
    """Ask on a thread and print what the turn cost."""
    final = await agent.ainvoke(
        {"messages": [{"role": "user", "content": question}]},
        thread_config(thread_id),
    )
    messages = final["messages"]
    new = messages[_SEEN.get(thread_id, 0):]
    _SEEN[thread_id] = len(messages)

    for m in new:
        for call in getattr(m, "tool_calls", []) or []:
            print(f"  tool call: {call['name']}({call['args']})")

    usage = _usage(new)
    calls = sum(len(getattr(m, "tool_calls", []) or []) for m in new)
    print(
        f"[transcript] {len(new)} new message(s), {calls} tool call(s) - "
        f"tokens in={usage['input']} out={usage['output']} total={usage['total']}"
    )
    return _text(messages[-1].content)


## A broad question

Nothing is taken away, so the model picks its own path. Watch the printed tool
calls: does it start with `list_services`, and does it reach for discovery before
querying Prometheus or Loki?


In [ ]:
print(await ask("Is anything wrong in the lab right now?"))


## A follow-up on the same thread

This is the memory test. The question names nothing — no container, no metric —
so it is only answerable if the checkpointed thread carried the previous turn
forward.


In [ ]:
print(await ask("Why? Show me the evidence for that."))


## A clean thread

A different `thread_id` shares nothing, which is what keeps two Discord channels
apart. The same question that just worked should now fall flat.


In [ ]:
print(await ask("Why? Show me the evidence for that.", thread_id="fresh"))


## Starting over

There is no external API for dropping a thread — the docs define `thread_id` as
the only handle on a conversation. To re-run this notebook from a clean slate,
use a new `thread_id`, or delete `data/memory.db` while the bot is stopped.
